# Baseline: Multinomial Logistic Regression

Train a multinomial logistic regression baseline to predict the FOMC decision class: `{lower, same, higher}`, and output predicted probabilities for each class.

- Uses walk-forward expanding window evaluation.
- No random split to avoid data leakage.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, log_loss, confusion_matrix
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Load data
_PARQUET_URL = 'https://raw.githubusercontent.com/echochoho1010/forecasting_fed_rate/master/data/df_model.parquet'
_CSV_URL     = 'https://raw.githubusercontent.com/echochoho1010/forecasting_fed_rate/master/data/df_model.csv'

try:
    df = pd.read_parquet(_PARQUET_URL)
except Exception:
    df = pd.read_csv(_CSV_URL)

# Sort by meeting_date to strictly enforce chronological order
df['meeting_date'] = pd.to_datetime(df['meeting_date'])
df = df.sort_values('meeting_date').reset_index(drop=True)

display(df.head())
print("Class distribution:\n", df['decision'].value_counts())

In [ ]:
# Prepare X and y
y = df['decision']

# Exclude non-feature columns
exclude_cols = ['meeting_date', 'decision', 'decision_num']
feature_cols = [c for c in df.columns if c not in exclude_cols]

# Select numeric features only
X = df[feature_cols].select_dtypes(include=[np.number])

# Forward fill NaNs if any
X = X.fillna(method='ffill').fillna(0)


In [ ]:
# Walk-forward evaluation (Expanding Window)
MIN_TRAIN = 30

preds_prob = []
preds_class = []
actual_labels = []
date_labels = []

classes = ['lower', 'same', 'higher']

for t in range(MIN_TRAIN, len(df)):
    X_train = X.iloc[:t]
    y_train = y.iloc[:t]
    
    X_test = X.iloc[[t]]
    y_test = y.iloc[t]
    
    # Pipeline: scale -> multinomial logit
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(multi_class='multinomial', solver='lbfgs', C=1.0, max_iter=500))
    ])
    
    # Fit on history [0..t-1]
    pipe.fit(X_train, y_train)
    
    # Predict observation t
    prob = pipe.predict_proba(X_test)[0]
    pred = pipe.predict(X_test)[0]
    
    # Extract probabilities matching the fixed ['lower', 'same', 'higher'] order
    pipe_classes = list(pipe.classes_)
    prob_dict = dict(zip(pipe_classes, prob))
    
    preds_prob.append([prob_dict.get(c, 0.0) for c in classes])
    preds_class.append(pred)
    actual_labels.append(y_test)
    date_labels.append(df.iloc[t]['meeting_date'])


In [ ]:
preds_prob = np.array(preds_prob)

# Report metrics
acc = accuracy_score(actual_labels, preds_class)
ll = log_loss(actual_labels, preds_prob, labels=classes)
cm = confusion_matrix(actual_labels, preds_class, labels=classes)

print(f"Accuracy: {acc:.4f}")
print(f"Log Loss: {ll:.4f}\n")

print("Confusion Matrix:")
df_cm = pd.DataFrame(cm, index=[f"True {c}" for c in classes], columns=[f"Pred {c}" for c in classes])
display(df_cm)


In [ ]:
# Create results table
results_df = pd.DataFrame({
    'meeting_date': date_labels,
    'true_label': actual_labels,
    'pred_label': preds_class,
    'P(lower)': preds_prob[:, 0],
    'P(same)': preds_prob[:, 1],
    'P(higher)': preds_prob[:, 2]
})

display(results_df.head())

# Save to disk (optional — gracefully skipped if directory does not exist)
import os
try:
    output_dir  = '../data'
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, 'preds_multinomial.csv')
    results_df.to_csv(output_path, index=False)
    print(f"Saved prediction table to {output_path}")
except Exception as e:
    print(f"Note: could not save to disk ({e}). Results displayed above.")